# Homomorphic Time-Lock Puzzles (HTLP)

[cite_start]Time-lock puzzles allow one to encrypt messages for the future, by efficiently generating a puzzle with a solutions that remains hidden until time $\mathcal{T}$ has elapsed. [cite: 5] [cite_start]The concept of homomorphic time-lock puzzles (HTLPs) extends this by allowing anyone to evaluate functions over these puzzles without solving them. [cite: 7] 

This notebook implements the **Linearly Homomorphic** scheme described in the paper:
* [cite_start]**Setup**: Generates the public parameters, including an RSA modulus $N$ and a time-hardness parameter $\mathcal{T}$. [cite: 282, 285] [cite_start]It computes $h := g^{2^\mathcal{T}}$, which can be optimized by the creator using the secret group order. [cite: 284]
* [cite_start]**Generate (PGen)**: Encrypts a secret $s$ using a random value $r$. [cite: 286, 288] [cite_start]It generates elements $u := g^r \pmod N$ and $v := h^{r \cdot N} \cdot (1+N)^s \pmod{N^2}$. [cite: 289]
* [cite_start]**Evaluate (PEval)**: Homomorphically adds secrets by multiplying the puzzle components $\tilde{u} := \prod u_i \pmod N$ and $\tilde{v} := \prod v_i \pmod{N^2}$. [cite: 300]
* [cite_start]**Solve (PSolve)**: Solves the puzzle by computing $w := u^{2^\mathcal{T}} \pmod N$ through inherently sequential repeated squaring. [cite: 294] [cite_start]The secret is recovered by computing $\frac{(v / w^N \pmod{N^2}) - 1}{N}$. [cite: 295]

In [ ]:
import random
import time
from sympy import isprime, randprime

def generate_strong_prime(bits):
    """Generates a strong prime p where p = 2p' + 1 and p' is also prime."""
    while True:
        p_prime = randprime(2**(bits-2), 2**(bits-1))
        p = 2 * p_prime + 1
        if isprime(p):
            return p, p_prime

def mod_inverse(a, m):
    """Computes the modular inverse of a modulo m."""
    return pow(a, -1, m)

class LHTLP:
    def __init__(self, bits=16, T=10):
        self.T = T
        # HP.PSetup
        p, p_prime = generate_strong_prime(bits)
        q, q_prime = generate_strong_prime(bits)
        self.N = p * q
        self.N2 = self.N ** 2
        
        # Order of the group J_N is phi(N)/2 for strong RSA integers
        self.phi_N = (p - 1) * (q - 1)
        self.order_JN = self.phi_N // 2
        
        # Sample a generator g for J_N
        while True:
            g_tilde = random.randrange(2, self.N)
            self.g = (-pow(g_tilde, 2, self.N)) % self.N
            if self.g > 1:
                break
                
        # Fast computation of h by the setup authority using the trapdoor (order_JN)
        exp = pow(2, self.T, self.order_JN)
        self.h = pow(self.g, exp, self.N)
        
    def PGen(self, s):
        """Generates a puzzle for secret s."""
        r = random.randrange(1, self.N2)
        u = pow(self.g, r, self.N)
        
        # v = h^(r*N) * (1+N)^s mod N^2
        h_rN = pow(self.h, r * self.N, self.N2)
        one_plus_N_s = pow(1 + self.N, s, self.N2)
        v = (h_rN * one_plus_N_s) % self.N2
        
        return (u, v)
        
    def PEval(self, puzzles):
        """Homomorphically evaluates the sum of multiple puzzles."""
        u_tilde = 1
        v_tilde = 1
        for u, v in puzzles:
            u_tilde = (u_tilde * u) % self.N
            v_tilde = (v_tilde * v) % self.N2
        return (u_tilde, v_tilde)
        
    def PSolve(self, puzzle):
        """Solves the puzzle by performing T sequential squarings."""
        u, v = puzzle
        
        # w = u^(2^T) mod N computed by repeated squaring
        w = u
        for _ in range(self.T):
            w = pow(w, 2, self.N)
            
        # s = ((v / w^N mod N^2) - 1) / N
        w_N = pow(w, self.N, self.N2)
        w_N_inv = mod_inverse(w_N, self.N2)
        
        numerator = (v * w_N_inv) % self.N2
        s = (numerator - 1) // self.N
        return s

### Part 1: Toy Example to Explain the Concept
Here we use very small prime numbers and a tiny time parameter $\mathcal{T}$ so the process executes instantly. We will encrypt two numbers, 5 and 10, add their puzzles homomorphically, and solve the resulting puzzle to get 15.

In [ ]:
print("--- TOY EXAMPLE ---")
# Very small numbers: 16-bit primes, T = 10 squarings
htlp_toy = LHTLP(bits=16, T=10)

print(f"Public Parameters:\nN = {htlp_toy.N}\nT = {htlp_toy.T}\n")

# Encrypt secrets
s1 = 5
s2 = 10
print(f"Secret 1: {s1}")
print(f"Secret 2: {s2}")

Z1 = htlp_toy.PGen(s1)
Z2 = htlp_toy.PGen(s2)
print(f"Puzzle 1 (Z1): {Z1}")
print(f"Puzzle 2 (Z2): {Z2}\n")

# Homomorphic Addition
Z_sum = htlp_toy.PEval([Z1, Z2])
print(f"Homomorphically Evaluated Puzzle (Z_sum): {Z_sum}\n")

# Solve the Evaluated Puzzle
start_time = time.time()
recovered_sum = htlp_toy.PSolve(Z_sum)
end_time = time.time()

print(f"Recovered Sum: {recovered_sum}")
print(f"Solved in: {end_time - start_time:.6f} seconds")

### Part 2: Realistic 5-Minute Time-Lock
To create a puzzle that takes roughly 5 minutes to solve, we need to calibrate the hardness parameter $\mathcal{T}$. Because Python handles large integer math differently depending on your CPU, we first run a quick 1-second calibration to find out how many squarings your computer does per second. We then scale this to 5 minutes (300 seconds).

*Note: For a true cryptographic environment, much larger primes (e.g., 1024-bit) would be used, but we use 256-bit here so the notebook remains responsive during the key-generation phase.*

In [ ]:
print("--- 5-MINUTE TIME-LOCK EXAMPLE ---")

# Calibration step to estimate squarings per second
test_N = generate_strong_prime(256)[0] * generate_strong_prime(256)[0]
test_val = random.randrange(2, test_N)

squarings = 0
calib_start = time.time()
# Run for 1 second to calibrate
while time.time() - calib_start < 1.0:
    test_val = pow(test_val, 2, test_N)
    squarings += 1

print(f"Calibration: Your CPU performs ~{squarings} squarings per second.")

# Target 5 minutes (300 seconds)
target_seconds = 300
T_5min = squarings * target_seconds
print(f"Setting T = {T_5min} for a ~5 minute delay.\n")

print("Initializing Setup (finding large primes)...")
htlp_real = LHTLP(bits=256, T=T_5min)

# Voting / Tallying Scenario
vote1 = 150 # Candidate A votes from district 1
vote2 = 350 # Candidate A votes from district 2

print("Generating Puzzles...")
Z1_real = htlp_real.PGen(vote1)
Z2_real = htlp_real.PGen(vote2)

print("Homomorphically evaluating the tally...")
Z_tally = htlp_real.PEval([Z1_real, Z2_real])

print(f"\nSolving the puzzle. This should take approximately 5 minutes. Please wait...")
solve_start = time.time()
recovered_tally = htlp_real.PSolve(Z_tally)
solve_end = time.time()

print(f"\nRecovered Tally: {recovered_tally} (Expected: {vote1 + vote2})")
print(f"Actual time taken to solve: {solve_end - solve_start:.2f} seconds")